## Fairness + Budget

In [21]:
# Objective: Maximize social coverage index under fairness constraint and $100M budget

import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB, quicksum

# -------------------------------------------------------------------
# 1. Load data
# -------------------------------------------------------------------
POP_PATH = "./ChildCareDeserts_Data/population.csv"
INC_PATH = "./ChildCareDeserts_Data/avg_individual_income.csv"
EMP_PATH = "./ChildCareDeserts_Data/employment_rate.csv"
REG_PATH = "./ChildCareDeserts_Data/child_care_regulated.csv"

BUDGET = 100_000_000

FACILITY = {
    "S": (100, 65_000),
    "M": (200, 95_000),
    "L": (400, 115_000),
}  # (capacity, cost)


def norm_zip(df):
    zc = [c for c in df.columns if "zip" in c.lower()][0]
    df = df.rename(columns={zc: "ZIP"})
    df["ZIP"] = df["ZIP"].astype(str).str.zfill(5)
    return df


pop = norm_zip(pd.read_csv(POP_PATH))
inc = norm_zip(pd.read_csv(INC_PATH))
emp = norm_zip(pd.read_csv(EMP_PATH))
reg = norm_zip(pd.read_csv(REG_PATH))

# Aggregate existing capacity by ZIP
cap_cols = [c for c in reg.columns if "capacity" in c.lower()]
reg["capacity"] = reg[cap_cols].sum(axis=1, numeric_only=True)
zip_cap = (
    reg.groupby("ZIP", as_index=False)["capacity"]
    .sum()
    .rename(columns={"capacity": "current_cap"})
)

# Simplify population
pop["pop_0_5"] = pop.filter(regex="0-5|under 5|0–5").sum(axis=1)
pop["pop_5_12"] = pop.filter(regex="5-9|10-14|5–9|10–14").sum(axis=1)
pop["pop_0_12"] = pop["pop_0_5"] + pop["pop_5_12"]

# Merge
zip_df = (
    pop.merge(zip_cap, on="ZIP", how="left")
    .merge(emp, on="ZIP", how="left")
    .merge(inc, on="ZIP", how="left")
    .fillna(0)
)

# High-demand flag
zip_df["high_demand"] = (zip_df.iloc[:, -2] >= 0.6) | (zip_df.iloc[:, -1] <= 60000)

# -------------------------------------------------------------------
# 2. Model
# -------------------------------------------------------------------
m = gp.Model("Fairness")

ZIPs = zip_df["ZIP"].tolist()

# Decision: number of facilities built
n_build = {
    (z, f): m.addVar(vtype=GRB.INTEGER, lb=0, name=f"build_{z}_{f}")
    for z in ZIPs
    for f in FACILITY
}

# Compute new capacities
new_cap = {z: quicksum(n_build[z, f] * FACILITY[f][0] for f in FACILITY) for z in ZIPs}

# Coverage
cov05, cov012, covw = {}, {}, {}
for i, row in zip_df.iterrows():
    z = row["ZIP"]
    pop05 = max(row["pop_0_5"], 1)
    pop012 = max(row["pop_0_12"], 1)
    cur = row["current_cap"]
    cov05[z] = (0.5 * cur + 0.5 * new_cap[z]) / pop05  # assume half slots for 0–5
    cov012[z] = (cur + new_cap[z]) / pop012
    covw[z] = (2 / 3) * cov05[z] + (1 / 3) * cov012[z]

# Fairness gap variables
min_cov = m.addVar(lb=0)
max_cov = m.addVar(lb=0)
for z in ZIPs:
    m.addConstr(covw[z] >= min_cov)
    m.addConstr(covw[z] <= max_cov)
m.addConstr(max_cov - min_cov <= 0.1)

# Desert elimination constraints
for i, row in zip_df.iterrows():
    z = row["ZIP"]
    if row["high_demand"]:
        m.addConstr(cov012[z] >= 0.5)
    else:
        m.addConstr(cov012[z] >= 1 / 3)
    m.addConstr(cov05[z] >= 2 / 3)

# Budget
total_cost = quicksum(n_build[z, f] * FACILITY[f][1] for z in ZIPs for f in FACILITY)
m.addConstr(total_cost <= BUDGET)

# Objective: maximize weighted social coverage
social_index = quicksum((2 / 3) * cov05[z] + (1 / 3) * cov012[z] for z in ZIPs)
m.setObjective(social_index, GRB.MAXIMIZE)

# -------------------------------------------------------------------
# 3. Solve
# -------------------------------------------------------------------
m.Params.OutputFlag = 1
m.optimize()

# -------------------------------------------------------------------
# 4. Results
# -------------------------------------------------------------------
if m.Status == GRB.OPTIMAL:
    print(f"\nTotal cost used: {total_cost.getValue():,.0f}")
    print(f"Fairness gap: {(max_cov.X - min_cov.X):.3f}")
    print(
        f"Average weighted coverage: {np.mean([covw[z].getValue() for z in ZIPs]):.3f}"
    )
else:
    print(
        "⚠️ Model infeasible — fairness or desert elimination may be too strict under $100M."
    )


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G231)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6586 rows, 4940 columns and 27984 nonzeros
Model fingerprint: 0x632e8ec0
Variable types: 2 continuous, 4938 integer (0 binary)
Coefficient statistics:
  Matrix range     [4e-03, 1e+05]
  Objective range  [6e-03, 3e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-04, 1e+08]
Presolve removed 1057 rows and 0 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.01 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible or unbounded
Best objective -, best bound -, gap -
⚠️ Model infeasible — fairness or desert elimination may be too strict under $100M.


## Minimum budget to achieve fairness

In [22]:
import math
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB, quicksum

POP_PATH = "./ChildCareDeserts_Data/population.csv"
INC_PATH = "./ChildCareDeserts_Data/avg_individual_income.csv"
EMP_PATH = "./ChildCareDeserts_Data/employment_rate.csv"
REG_PATH = "./ChildCareDeserts_Data/child_care_regulated.csv"

# New-build options (ZIP-level, no spatial constraints)
FACILITY_OPTIONS = [
    {"name": "S", "cap_total": 100, "cap_05_max": 50, "cost": 65000},
    {"name": "M", "cap_total": 200, "cap_05_max": 100, "cost": 95000},
    {"name": "L", "cap_total": 400, "cap_05_max": 200, "cost": 115000},
]


# ---------------------------
# Helpers
# ---------------------------
def normalize_zip(df):
    for c in df.columns:
        if "zip" in c.lower():
            df = df.rename(columns={c: "ZIP"})
            break
    df["ZIP"] = df["ZIP"].astype(str).str.zfill(5)
    return df


def safe_sum_cols(df, cols):
    return df[cols].sum(axis=1, numeric_only=True) if len(cols) > 0 else 0.0


def load_and_prepare():
    pop_df = pd.read_csv(POP_PATH)
    inc_df = pd.read_csv(INC_PATH)
    emp_df = pd.read_csv(EMP_PATH)
    reg_df = pd.read_csv(REG_PATH)

    pop_df = normalize_zip(pop_df)
    inc_df = normalize_zip(inc_df)
    emp_df = normalize_zip(emp_df)
    reg_df = normalize_zip(reg_df)

    # Population buckets → 0–12
    c_0_5 = [
        c
        for c in pop_df.columns
        if any(k in c.lower() for k in ["0-5", "0–5", "under 5"])
    ]
    c_5_9 = [c for c in pop_df.columns if any(k in c.lower() for k in ["5-9", "5–9"])]
    c_10_14 = [
        c for c in pop_df.columns if any(k in c.lower() for k in ["10-14", "10–14"])
    ]

    pop_df["pop_0_5"] = safe_sum_cols(pop_df, c_0_5)
    pop_df["pop_5_9"] = safe_sum_cols(pop_df, c_5_9)
    pop_df["pop_10_14"] = safe_sum_cols(pop_df, c_10_14)
    pop_df["pop_0_12"] = (
        pop_df["pop_0_5"] + pop_df["pop_5_9"] + 0.4 * pop_df["pop_10_14"]
    )

    inc_df = inc_df.rename(columns={"average income": "average_income"})
    emp_df = emp_df.rename(columns={"employment rate": "employment_rate"})

    # Facility capacities
    if "total_capacity" in reg_df.columns:
        reg_df["fac_capacity"] = reg_df["total_capacity"]
    else:
        cap_cols = [c for c in reg_df.columns if "capacity" in c.lower()]
        if len(cap_cols) == 0:
            raise ValueError("No capacity columns found in child_care_regulated.csv")
        reg_df["fac_capacity"] = reg_df[cap_cols].sum(axis=1, numeric_only=True)

    u5_cols = [
        c
        for c in reg_df.columns
        if any(k in c.lower() for k in ["0-5", "0–5", "infant", "toddler", "preschool"])
    ]
    reg_df["fac_u5_capacity"] = (
        reg_df[u5_cols].sum(axis=1, numeric_only=True) if len(u5_cols) > 0 else 0.0
    )

    # Per-ZIP current totals
    zip_total = (
        reg_df.groupby("ZIP", as_index=False)["fac_capacity"]
        .sum()
        .rename(columns={"fac_capacity": "current_total"})
    )
    zip_u5 = (
        reg_df.groupby("ZIP", as_index=False)["fac_u5_capacity"]
        .sum()
        .rename(columns={"fac_u5_capacity": "current_u5"})
    )

    zip_df = pd.DataFrame({"ZIP": sorted(set(pop_df["ZIP"]))})
    zip_df = (
        zip_df.merge(zip_total, on="ZIP", how="left")
        .merge(zip_u5, on="ZIP", how="left")
        .merge(pop_df[["ZIP", "pop_0_5", "pop_0_12"]], on="ZIP", how="left")
        .merge(emp_df[["ZIP", "employment_rate"]], on="ZIP", how="left")
        .merge(inc_df[["ZIP", "average_income"]], on="ZIP", how="left")
    )
    fill_cols = [
        "current_total",
        "current_u5",
        "pop_0_5",
        "pop_0_12",
        "employment_rate",
        "average_income",
    ]
    zip_df[fill_cols] = zip_df[fill_cols].fillna(0.0)

    return pop_df, inc_df, emp_df, reg_df, zip_df


def build_model(zip_df, reg_df, fairness_target=0.1, eps=1e-4):
    m = gp.Model("Eliminate_Deserts_Fairness")
    m.Params.OutputFlag = 0

    ZIPs = zip_df["ZIP"].tolist()

    # --- New builds per ZIP (S/M/L)
    n_build = {
        (z, opt["name"]): m.addVar(
            vtype=GRB.INTEGER, lb=0, name=f"build_{z}_{opt['name']}"
        )
        for z in ZIPs
        for opt in FACILITY_OPTIONS
    }
    new_total = {
        z: quicksum(
            n_build[(z, opt["name"])] * opt["cap_total"] for opt in FACILITY_OPTIONS
        )
        for z in ZIPs
    }
    new_u5_cap = {
        z: quicksum(
            n_build[(z, opt["name"])] * opt["cap_05_max"] for opt in FACILITY_OPTIONS
        )
        for z in ZIPs
    }
    new_u5 = {z: m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"new_u5_{z}") for z in ZIPs}
    for z in ZIPs:
        m.addConstr(new_u5[z] <= new_u5_cap[z], name=f"cap_u5_new_{z}")

    # --- Expansions per facility (≤ 20%, 3 segments)
    reg_df = (
        reg_df.reset_index(drop=True).reset_index().rename(columns={"index": "f_id"})
    )
    F = [
        int(f)
        for f, cap in zip(reg_df["f_id"], reg_df["fac_capacity"])
        if (cap and cap > 0)
    ]
    fac_zip = {f: reg_df.loc[f, "ZIP"] for f in F}
    n_f = {f: float(reg_df.loc[f, "fac_capacity"]) for f in F}

    y1, y2, y3, x_total, x_u5 = {}, {}, {}, {}, {}
    for f in F:
        nf = n_f[f]
        y1[f] = m.addVar(lb=0, ub=0.10 * nf, vtype=GRB.CONTINUOUS, name=f"y1_{f}")
        y2[f] = m.addVar(lb=0, ub=0.05 * nf, vtype=GRB.CONTINUOUS, name=f"y2_{f}")
        y3[f] = m.addVar(lb=0, ub=0.05 * nf, vtype=GRB.CONTINUOUS, name=f"y3_{f}")
        x_total[f] = m.addVar(lb=0, ub=0.20 * nf, vtype=GRB.CONTINUOUS, name=f"x_{f}")
        x_u5[f] = m.addVar(lb=0, ub=0.20 * nf, vtype=GRB.CONTINUOUS, name=f"x_u5_{f}")
        m.addConstr(x_total[f] == y1[f] + y2[f] + y3[f], name=f"sum_seg_{f}")
        m.addConstr(x_u5[f] <= x_total[f], name=f"u5_le_total_{f}")

    exp_total = {z: quicksum(x_total[f] for f in F if fac_zip[f] == z) for z in ZIPs}
    exp_u5_zip = {z: quicksum(x_u5[f] for f in F if fac_zip[f] == z) for z in ZIPs}

    # --- Coverage ratios per ZIP
    cov_tot, cov_u5, cov_w = {}, {}, {}
    for _, row in zip_df.iterrows():
        z = row["ZIP"]
        curT = float(row["current_total"])
        curU = float(row["current_u5"])
        p12 = float(row["pop_0_12"])
        p05 = float(row["pop_0_5"])
        d12 = max(p12, 1.0)
        d05 = max(p05, 1.0)

        cov_tot[z] = (curT + exp_total[z] + new_total[z]) / d12
        cov_u5[z] = (curU + exp_u5_zip[z] + new_u5[z]) / d05
        cov_w[z] = (2.0 / 3.0) * cov_u5[z] + (1.0 / 3.0) * cov_tot[z]

    # --- DESERT ERADICATION constraints (per prompt)
    # High-demand if (employment_rate >= 60%) OR (average_income <= $60k)
    emp = dict(zip(zip_df["ZIP"], zip_df["employment_rate"]))
    inc = dict(zip(zip_df["ZIP"], zip_df["average_income"]))

    for z in ZIPs:
        is_high = (emp[z] >= 0.60) or (inc[z] <= 60000.0)
        thr = 0.50 if is_high else (1.0 / 3.0)
        # Total coverage must strictly exceed threshold
        m.addConstr(cov_tot[z] >= thr + eps, name=f"desert_tot_{z}")
        # NYS policy: 0–5 coverage must be at least 2/3 of pop(0–5)
        m.addConstr(cov_u5[z] >= (2.0 / 3.0) + eps, name=f"u5_policy_{z}")

    # --- Fairness band on weighted coverage (gap <= 0.1)
    min_cov = m.addVar(lb=0, name="min_cov_w")
    max_cov = m.addVar(lb=0, name="max_cov_w")
    for z in ZIPs:
        m.addConstr(cov_w[z] >= min_cov, name=f"min_cov_{z}")
        m.addConstr(cov_w[z] <= max_cov, name=f"max_cov_{z}")
    gap = m.addVar(lb=0, name="fairness_gap")
    m.addConstr(gap == max_cov - min_cov, name="gap_def")
    m.addConstr(gap <= float(fairness_target), name="gap_target")

    # --- Costs (piecewise expansion + $100/slot for 0–5, builds from table)
    exp_terms = []
    for f in F:
        nf = n_f[f]
        c1 = (20000.0 + 200.0 * nf) / nf
        c2 = (20000.0 + 400.0 * nf) / nf
        c3 = (20000.0 + 1000.0 * nf) / nf
        exp_terms += [c1 * y1[f], c2 * y2[f], c3 * y3[f], 100.0 * x_u5[f]]

    expansion_cost = quicksum(exp_terms)
    build_cost = quicksum(
        n_build[(z, opt["name"])] * opt["cost"]
        for z in ZIPs
        for opt in FACILITY_OPTIONS
    )
    new_u5_cost = quicksum(100.0 * new_u5[z] for z in ZIPs)
    total_cost = expansion_cost + build_cost + new_u5_cost

    # --- Objective: minimize total cost (no budget)
    m.setObjective(total_cost, GRB.MINIMIZE)

    return m, {
        "ZIPs": ZIPs,
        "cov_tot": cov_tot,
        "cov_u5": cov_u5,
        "cov_w": cov_w,
        "gap": gap,
        "min_cov": min_cov,
        "max_cov": max_cov,
        "total_cost": total_cost,
    }


# ---------------------------
# Solve
# ---------------------------
pop_df, inc_df, emp_df, reg_df, zip_df = load_and_prepare()

m, R = build_model(zip_df, reg_df, fairness_target=0.1, eps=1e-4)
m.Params.OutputFlag = 1  # show logs
m.optimize()

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G231)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 39438 rows, 84602 columns and 210220 nonzeros
Model fingerprint: 0xb2c6d293
Variable types: 79664 continuous, 4938 integer (0 binary)
Coefficient statistics:
  Matrix range     [2e-05, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [2e-01, 2e+02]
  RHS range        [1e-04, 3e+02]
Presolve removed 21444 rows and 49555 columns
Presolve time: 0.15s
Presolved: 17994 rows, 35047 columns, 89478 nonzeros
Variable types: 30475 continuous, 4572 integer (0 binary)

Root relaxation: objective 2.433663e+11, 9423 iterations, 0.04 seconds (0.05 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 2.4337e+11    0 1523          - 

In [23]:
# -------------------------------------------------------
# Extraction of results
# -------------------------------------------------------

if m.SolCount > 0:
    print("\n=== MODEL RESULTS ===")
    print(f"Total cost: ${m.objVal:,.2f}")
    print(f"Fairness gap: {R['gap'].X:.4f}\n")


=== MODEL RESULTS ===
Total cost: $243,421,581,576.70
Fairness gap: 0.1000

